In [1]:
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit.library import QFTGate, grover_operator, UnitaryGate
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

import random
import numpy as np
import time
import os
from dotenv import load_dotenv

In [2]:
class GroverAdaptiveSearch:
    """
        Grover Adaptive Search (GAS) implementation for Higher-Order Unconstrained Binary Optimization (HUBO).
        Main reference: https://doi.org/10.1109/TCOMM.2023.3244924

        Attributes:
            n (int): Number of binary variables.
            m (int): Number of qubits for representing the objective function.
            hubo (dict): HUBO formulation of the problem.
    """
    def __init__(self, n, m, hubo, backend, trans_opt_level):
        self.n = n
        self.m = m
        self.hubo = hubo
        self.qc = QuantumCircuit(n+m)
        self.backend = backend
        self.trans_opt_level = trans_opt_level

        # GAS setup
        self.i = 0
        self.k = 1
        self.b = [random.randint(0, 1) for _ in range(self.n)]
        self.y = self.hubo['constant']

        # Evaluate objective function
        for term in self.hubo['terms']:
            res = term['coef']
            for var in term['vars']:
                res *= self.b[var]
            self.y += res

        self.L = random.randint(0, self.k)
        self.lambda_ = 8/7

        # metrics
        self.transpilation_time = []

        # checkpoint variables
        self.checkpoint = {
            'i':None,
            'k':None,
            'b':None,
            'y':None,
            'L':None,
            'transpilation_time':None
        } 
    
    def build_A_y_i(self):
        A_y_i = QuantumCircuit(self.n + self.m, name='A_y_i')
        A_y_i.h(range(self.n + self.m))

        # encode constant
        encode_qc = QuantumCircuit(self.m, name=f"U_G({self.hubo['constant']} pi/{2**(self.m-1)})")
        angle = 2 * np.pi * (self.hubo['constant'] - self.y) / 2 ** self.m
        for j in range(self.m):
            factor = 2 ** j
            encode_qc.p(factor * angle, j)
        encode_gate = encode_qc.to_gate()
        A_y_i.append(encode_gate, range(self.n, self.n+self.m))

        # encode terms
        for term in self.hubo['terms']:
            encode_qc = QuantumCircuit(self.m, name=f"U_G({term['coef']} pi/{2**(self.m-1)})")
            angle = 2 * term['coef'] * np.pi / 2 ** self.m
            for j in range(self.m):
                factor = 2 ** j
                encode_qc.p(factor * angle, j)
            encode_gate = encode_qc.to_gate().control(len(term['vars']))
            A_y_i.append(encode_gate, term['vars'] + list(range(self.n, self.n+self.m)))

        # IQFT
        iqft = QFTGate(self.m).inverse()
        A_y_i.append(iqft, range(self.n, self.n+self.m))
        
        return A_y_i

    def build_D_n(self):
        D_n = QuantumCircuit(self.n, name='D_n')
        D_n.h(range(self.n))
        D_n.x(range(self.n))
        # Multi-controlled phase gate to flip the target state
        D_n.mcp(np.pi, list(range(self.n - 1)), self.n - 1)
        D_n.x(range(self.n))
        D_n.h(range(self.n))
        return D_n

    def build_G(self):
        G = QuantumCircuit(self.n + self.m, name='G')
        
        # 1. Oracle (Flip phase of negative states using the MSB of register m)
        G.z(self.n + self.m - 1)
        
        # 2. Uncompute State Preparation
        G.append(self.build_A_y_i().inverse(), range(self.n + self.m))
        
        # 3. Diffuse ONLY the n variable qubits
        # This automatically builds the H^(tensor n) -> (2|0><0| - I) -> H^(tensor n) chain
        diffusion_gate = grover_operator(oracle=QuantumCircuit(self.n)).to_gate()
        G.append(diffusion_gate, range(self.n)) # Tied strictly to the n register
        
        # 4. Re-compute State Preparation
        G.append(self.build_A_y_i(), range(self.n + self.m))
        
        return G

    def build(self):
        self.qc.append(self.build_A_y_i(), range(self.n + self.m))
        for i in range(self.L):
            self.qc.append(self.build_G(), range(self.n + self.m))

        self.qc.measure_all()

    def run(self, shots=1000):
        pm = generate_preset_pass_manager(backend=backend, optimization_level=self.trans_opt_level)

        start_trans_time = time.perf_counter()
        transpiled_qc = pm.run(self.qc)
        end_trans_time = time.perf_counter()

        transpilation_time = end_trans_time - start_trans_time
        print(f'Transpilation time: {transpilation_time:.5f}')
        self.transpilation_time.append(transpilation_time)
        
        sampler = SamplerV2(mode=self.backend)
        job = sampler.run([(transpiled_qc, None, shots)])
        
        results = job.result()
        counts = results[0].data.meas.get_counts()
        
        most_frequent_state = max(counts, key=counts.get)
        clean_state = most_frequent_state.replace(" ", "")
        measured_b_str = clean_state[-self.n:]
        new_b = [int(bit) for bit in measured_b_str]
        
        # Calculate true objective value classically
        new_y = self.hubo['constant']
        for term in self.hubo['terms']:
            res = term['coef']
            for var in term['vars']:
                res *= new_b[var]
            new_y += res
            
        return new_b, new_y

    def optimize(self, max_iter=20, shots=1024, verbose=True):
        """
        Executes the main loops of GAS until convergence or max_iter is hit.
        """
        if verbose:
            print(f"Starting GAS. Initial Threshold (y_0): {self.y} | State: {self.b}")
            print("-" * 60)

        for iteration in range(self.i + 1, max_iter):
            self.i = iteration
            
            # Step 1: Reassemble the circuit using the current loop parameters
            self.build()
            
            # Step 2: Sample from the quantum simulator
            candidate_b, candidate_y = self.run(shots=shots)
            
            # Step 3: Adaptive Update Rules (Step 6 of paper pseudocode)
            if candidate_y < self.y:
                if verbose:
                    print(f"Iteration {self.i:02d}: IMPROVEMENT FOUND!")
                    print(f"  Old Threshold: {self.y} -> New Threshold: {candidate_y}")
                    print(f"  Config: {candidate_b} | Grover Steps (L) Used: {self.L}")
                
                # Update baseline parameters
                self.b = candidate_b
                self.y = candidate_y
                self.k = 1  # Reset step parameter on success
            else:
                # No improvement found: widen the random exploration step range
                # Cap the growth of k at roughly 2^(n/2) to prevent exponential exploration overhead
                self.k = min(self.lambda_ * self.k, int(2**(self.n / 2)) + 1)
                if verbose:
                    print(f"Iteration {self.i:02d}: No improvement. Expanding search range (k={self.k:.2f})")
            
            # Choose a new random number of Grover iterations for the next cycle
            self.L = random.randint(0, int(self.k))
            print(f'Iteration Count: {self.L}')

            # Checkpoint
            self.checkpoint = {
                'i':self.i,
                'k':self.k,
                'b':self.b,
                'y':self.y,
                'L':self.L,
                'transpilation_time':self.transpilation_time
            }
            
        if verbose:
            print("-" * 60)
            print(f"Optimization Finished. Global Minimum found: {self.y} at configuration {self.b}")
            
        return self.b, self.y

    def save_progress(self):
        return self.checkpoint

    def load_progress(self, checkpoint):
        self.i = checkpoint['i']
        self.k = checkpoint['k']
        self.b = checkpoint['b']
        self.y = checkpoint['y']
        self.L = checkpoint['L']
        self.transpilation_time = checkpoint['transpilation_time']

In [3]:
sample_data = {
    'constant': 1,
    'terms': [
        {'vars': [0], 'coef': 1},
        {'vars': [1, 2], 'coef': -2},
        {'vars': [1, 2, 3], 'coef': -1.8},
    ]
}

n = 4
m = 3
hubo = sample_data
backend = AerSimulator()
GAS = GroverAdaptiveSearch(n, m, hubo, backend, 1)
best_config, minimum_energy = GAS.optimize(max_iter=20)

Starting GAS. Initial Threshold (y_0): -1.0 | State: [0, 1, 1, 0]
------------------------------------------------------------
Transpilation time: 0.00182
Iteration 01: No improvement. Expanding search range (k=1.14)
Iteration Count: 0
Transpilation time: 0.00158
Iteration 02: No improvement. Expanding search range (k=1.31)
Iteration Count: 1
Transpilation time: 0.00329
Iteration 03: No improvement. Expanding search range (k=1.49)
Iteration Count: 1
Transpilation time: 0.00579
Iteration 04: No improvement. Expanding search range (k=1.71)
Iteration Count: 1
Transpilation time: 0.00591
Iteration 05: No improvement. Expanding search range (k=1.95)
Iteration Count: 1
Transpilation time: 0.00749
Iteration 06: No improvement. Expanding search range (k=2.23)
Iteration Count: 2
Transpilation time: 0.01149
Iteration 07: No improvement. Expanding search range (k=2.55)
Iteration Count: 2
Transpilation time: 0.01433
Iteration 08: No improvement. Expanding search range (k=2.91)
Iteration Count: 1
T

In [4]:
def preprocess_hubo_for_direct_encoding(raw_hubo, m):
    """
    Normalizes a real-valued HUBO problem to safely fit inside the phase bounds 
    of an m-qubit register, protecting against phase wrapping.
    
    Parameters:
        raw_hubo (dict): The original dictionary with 'constant' and 'terms'.
        m (int): Number of evaluation qubits you intend to use.
        
    Returns:
        dict: A scaled HUBO ready for the unmodified GAS implementation.
        float: The scale factor applied (needed to convert final energies back).
    """
    # 1. Calculate the absolute worst-case maximum energy bound
    max_energy_bound = abs(raw_hubo['constant']) + sum(abs(t['coef']) for t in raw_hubo['terms'])
    
    # 2. Calculate the maximum value a continuous two's complement register can hold
    # We use 2**(m-1) to preserve the negative/positive sign split boundary
    max_register_value = 2 ** (m - 1)
    
    # 3. Create our global scaling factor
    # Leave a tiny 1% safety margin to prevent boundary rounding issues
    scale_factor = (max_register_value / max_energy_bound) * 0.99
    
    # 4. Deep copy the dictionary structure so we don't mutate your original data
    processed_hubo = copy.deepcopy(raw_hubo)
    
    # 5. Apply the scale factor uniformly to all coefficients
    processed_hubo['constant'] *= scale_factor
    for term in processed_hubo['terms']:
        term['coef'] *= scale_factor
        
    return processed_hubo, scale_factor

In [5]:
import json
import ast
import copy

with open('sample2.json', 'r', encoding='utf-8') as file:
    data = json.load(file)[0]

n = data['num_variables']
m = 20
hubo = ast.literal_eval(data['objective_json'])
scaled_hubo, scale_factor = preprocess_hubo_for_direct_encoding(hubo, m)
print(f'Scaled HUBO by {scale_factor}')

Scaled HUBO by 5.684471074047787e-06


Connect to IBM Hardware

In [6]:
# Access the variables 
load_dotenv()
token = os.getenv("IBM_QP_TOKEN")
instance = os.getenv("IBM_QP_INSTANCE")

In [7]:
QiskitRuntimeService.save_account(
    token=token,
    instance=instance,
    overwrite=True
)

service = QiskitRuntimeService()
backend = service.least_busy(simulator=False, operational=True)

Details of the IBM Hardware

In [8]:
# Assuming 'backend' is your QiskitRuntimeService backend object
config = backend.configuration()

print(f"Backend Name:          {config.backend_name}")
print(f"Version:               {config.backend_version}")
print(f"Number of Qubits:      {config.n_qubits}")
print(f"Basis Gates:           {config.basis_gates}")
print(f"Processor Type:        {config.processor_type}")
print(f"Supported Operations:  {config.supported_instructions}")
print(f"Default Rep Delay:     {getattr(config, 'default_rep_delay', 'N/A')}")
print(f"Description:           {config.description}")
print(f"dt (Cycle Time):       {getattr(config, 'dt', 'N/A')}")
print(f"dtm (Measure Time):    {getattr(config, 'dtm', 'N/A')}")
print(f"Dynamic Reprate:       {getattr(config, 'dynamic_reprate_enabled', 'N/A')}")
print(f"Local Backend:         {config.local}")
print(f"Number of Registers:   {getattr(config, 'n_registers', 'N/A')}")
print(f"Rep Delay Range:       {getattr(config, 'rep_delay_range', 'N/A')}")
print(f"Sample Name:           {getattr(config, 'sample_name', 'N/A')}")

Backend Name:          ibm_kingston
Version:               1.0.0
Number of Qubits:      156
Basis Gates:           ['cz', 'id', 'rz', 'sx', 'x']
Processor Type:        {'family': 'Heron', 'revision': '2'}
Supported Operations:  ['cz', 'id', 'delay', 'measure', 'measure_2', 'reset', 'rz', 'sx', 'x', 'if_else', 'store']
Default Rep Delay:     0.00025
Description:           156 qubit device
dt (Cycle Time):       4e-09
dtm (Measure Time):    4e-09
Dynamic Reprate:       True
Local Backend:         False
Number of Registers:   1
Rep Delay Range:       [0.0, 0.002]
Sample Name:           family: Heron, revision: 2


`default_rep_delay` (seconds)
- pre-configured time delay that a quantum processor (QPU) automatically inserts between multiple circuit executions (shots) to allow qubits to cool down and reset to their ground state.

`dt` (seconds)
- The fundamental unit of time (in seconds) for a single sample on the qubit drive channels.
- For instance, if we apply $X$ gate, this is converted to one microwave pulse. This microwave pulse takes multiple steps, one step takes time `dt`

`dtm` (seconds)
- The fundamental unit of time (in seconds) for a single sample on the qubit readout channels.
- For instance, if we want to measure a qubit, this is converted to one readout microwave pulse. This microwave pulse takes multiple steps, one step takes time `dt`

`dynamic_reprate_enabled` (Boolean)
- tells you whether you are allowed to custom-configure the repetition rate (the delay time between experiments) when you send jobs to that specific quantum computer.
- When you run a quantum circuit, the computer applies gates and then measures the qubits. The measurement process puts a lot of energy into the system. After the measurement is done, the qubits are "excited" and messy. Before you can run the circuit again (for the next "shot"), you have to wait for the qubits to naturally cool down and relax back to their ground state ($|0\rangle$).The time you wait between shots is called the Repetition Delay (`rep_delay`).

`local` (Boolean)
- tells you where the quantum backend simulator or hardware is physically located and running relative to your local computer.

`n_qubits` (integer)
- number of qubits present in the hardware

`n_registers` (integer)
- tells you how many separate "slots" of classical memory the hardware has available to store mid-circuit measurement results and make real-time decisions.

`open_pulse` (Boolean)
- tells you whether the backend supports direct, low-level pulse control.
- it answers the question: Am I allowed to bypass abstract gates and design my own raw microwave pulses for this quantum computer?

`processor_type` (dict) and `sample_name` (string)
- tells you the details of the hardware processor

`rep_delay_range`
- defines the minimum and maximum allowed wait times (in seconds) that you can set between your circuit shots.

`supported_instructions`
- list of strings that names every single low-level command that the physical quantum processor is capable of executing natively.

In [ ]:
GAS = GroverAdaptiveSearch(n, m, scaled_hubo, backend, 1)
best_config, minimum_energy = GAS.optimize(max_iter=15)

Starting GAS. Initial Threshold (y_0): 6031.1577149100185 | State: [1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0]
------------------------------------------------------------
Transpilation time: 367.15574
